## STEP 1)Data Preparation & Model Loading
-Importing the Libraries
The first part imports the libraries needed for the pytorch project see it is used to create and work with the neural network while torchvision provides the VGG16 model image transformations and the CIFAR-10 dataset and dataLoader and random_split are used to organise the dataset into batches and divide the training data into separate sections the PIL library is included for working with images while requests and BytesIO can be used to retrieve and process an image from a URL torch.nn provides tools for creating and modifying neural network layers and torch.optim provides optimisation methods that will be used later during model training

In [2]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, random_split
from PIL import Image
import requests
from io import BytesIO
import torch.nn as nn
import torch.optim as optim

## 4.2 Choice of training model
This code allows the user to choose what training model they want to usem see the two available options are VGG16 and VGG19 the input() function asks the user to enter either 1 or 2 and an if statement checks their choice. The selected model is then loaded and a message confirms which model was chosen if the user enters anything other than 1 or 2 a ValueError is raised to show that the choice is invalid.

In [3]:
print("Choose a training model:")
print("1 - VGG16")
print("2 - VGG19")

choice = input("Enter your choice (1 or 2): ")

if choice == "1":
    model = models.vgg16(pretrained=True)
    print("VGG16 selected")
elif choice == "2":
    model = models.vgg19(pretrained=True)
    print("VGG19 selected")
else:
    raise ValueError("Invalid choice. Please enter 1 or 2.")

Choose a training model:
1 - VGG16
2 - VGG19


Enter your choice (1 or 2):  2


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 160MB/s]  


VGG19 selected


## Loading the Pre-trained VGG16 Model

The VGG16 model is loaded using the pre trained weights VGG16 is a convolutional neural network that has already been trained to recognise features in images using a pre trained model means that the model already has useful knowledge about visual features such as shapes edges and patterns this provides a starting point for working with the CIFAR-10 image dataset rather than training the entire model from the beginningthe model is stored in the variable `model` what allows it to be modified and used during the later stages off the project.


In [4]:
# Load the pre-trained VGG16 model
model = models.vgg16(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 179MB/s]  


### Modifying the classifier

The final classifier layer of the pre-trained VGG16 model is modified so that it can be used with the CIFAR-10 dataset and CIFAR-10 contains 10 different image classes so the final layer needs to produce 10 outputs the `nn.Linear(4096, 10)` layer takes the 4096 features produced by the previous layer and converts them into 10 output values see each out put represents one of the CIFAR-10 classes this allows the model to make predictions specifically for the 10 categories in the dataset.


In [5]:
# Modify the classifier to fit CIFAR-10
model.classifier[6] = nn.Linear(4096, 10)

###  Selecting the Computing Device

The code checks if a CUDA-compatible GPU is available on the computer then CUDA allows PyTorch to use a compatible NVIDIA GPU to perform calculations if CUDA is available the model will use the GPU otherwise the CPU will be used the selected device is stored in the `device` variable the `model.to(device)` command then moves the VGG16 model to the selected device this allows the model to run using the available hardware during the later training and inference stages.


In [6]:
# Select the GPU if available, otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

###  Freezing the Feature Extractor

The feature extractor layers off the VGG16 model are frozen so that there pre trained weights are not updated during training the `for` loop goes through the parameters in `model.features` and `requires_grad = False` prevents gradients from being calculated for these parameters this means the existing feature extraction knowledge in VGG16 is kept while the new classifier can learn how to use these features to classify the 10 different CIFAR-10 classes then freezing the feature extractor can also reduce the amount of computation required during training.


In [7]:
# Freeze the feature extractor layers
for param in model.features.parameters():
    param.requires_grad = False

###  Training Data Transformations

The training transformations prepare the CIFAR-10 images so that they can be used by the VGG16 model `RandomHorizontalFlip()` randomly flips some images horizontally while `RandomCrop()` creates slightly different versions off the images by cropping them with padding these transformations help introduce variation into the training data `Resize(224)` changes the image size to 224 pixels what is the required input size for the VGG16 model `ToTensor()` converts each image into a PyTorch tensor so it can be processed by the neural network finally, `Normalize()` standardises the image values using the specified mean and standard deviation values this prepares the images in a suitable format for the pre-trained VGG16 model


In [8]:
# Define transformations for the training data
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

### Test Data Transformations

The test data also needs to be transformed into a format that can be processed by the VGG16 model the images are first resized to 224 pixels so that they match the input size expected by VGG16 `ToTensor()` converts the images into PyTorch tensors the images are then normalised using the same mean and standard deviation values used for the training data unlike the training transformations andom flipping and cropping are not used because the test images should be evaluated in there original form without adding random changes


In [9]:
# Define transformations for the test data
transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

###  Loading the CIFAR-10 Training Dataset

The CIFAR-10 training dataset is loaded using the `CIFAR10` dataset provided by torchvision the `root` parameter specifies that the dataset will be stored in the `data` folder the `train=True` setting selects the training data rather than the test data the `download=True` setting allows the dataset to be downloaded automatically if it is not already stored on the computer the `transform_train` transformation is also applied to the training images so they are prepared in the correct format for the VGG16 model


In [10]:
# Load the CIFAR-10 training dataset
train_dataset = CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

100%|██████████| 170M/170M [43:11<00:00, 65.8kB/s] 


### Loading the CIFAR-10 Test Dataset 

The CIFAR-10 test dataset is loaded using the `CIFAR10` dataset from torchvision see the `root` parameter specifies that the dataset is stored in the `data` folder the `train=False` setting selects the test portion of the CIFAR-10 dataset instead of the training portion the `download=True` setting allows the dataset to be downloaded automatically if it is not already available. The `transform_test` transformation is applied to the test images so they are resized converted into tensors and normalised before they are used to evaluate the model


In [11]:
# Load the CIFAR-10 test dataset
test_dataset = CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

### Splitting the Training Dataset

The training dataset is divided into separate training and validation datasets the `train_size` is calculated as 80% of the total training dataset while the remaining data is used for validation the `random_split()` function then randomly divides the original training dataset using these two sizes the training data is used to train the model while the validation data can be used to check how well the model is performing during training.

In [12]:
# Split the training dataset into training and validation sets
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size]
)

In [13]:
# Load the CIFAR-10 training dataset
train_dataset = CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

###  Creating the Training DataLoader

The `DataLoader` is created to provide the training data to the model in smaller batches rather than loading the entire dataset at once the `train_dataset` contains the training images and there labels the `batch_size=64` setting means that 64 images are processed at a time `shuffle=True` randomly changes the order off the training data between passes through the dataset this helps prevent the model from relying on the order in what the training examples are provided `num_workers=4` allows four worker processes to help load the data.


In [14]:
# Create the training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4
)

### Creating the Validation DataLoader

The validation `DataLoader` is created to provide the validation dataset to the model in batches the `val_dataset` contains the 20% of the original training data that was separated for validation the `batch_size=64` setting means that the validation images are processed in groups of 64 `shuffle=False` keeps the validation data in a consistent order because the validation data is being used to evaluate the model rather than train it the `num_workers=4` setting allows four worker processes to help load the data


In [15]:
# Create the validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

###  Creating the Test DataLoader

The test `DataLoader` is created to provide the CIFAR-10 test dataset to the model in batches the `test_dataset` contains the separate test images that will be used to evaluate the trained model the `batch_size=64` setting means that the images are processed in groups of 64 `shuffle=False` keeps the test data in a consistent order because the data is being used for evaluation rather than training the `num_workers=4` setting provides four worker processes to help load the data efficiently


In [16]:
# Create the test DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

## Step 2) Training

In this step I trained the VGG16 model using a cross entropy loss function and the SGD optimiser the optimiser updates the final classifier layer off the model so that it can learn to classify the 10 different CIFAR-10 classes the training is carried out for two epochs with the images processed in batches from the training data loader and during each batch the model makes predictions calculates the loss performs backpropagation and updates the model's weights after each epoch the model is tested using the validation dataset to calculate the validation accuracy and check how well it is learning

### Defining the Optimiser and Learning Rate Scheduler

The optimiser is responsible for updating the trainable parameters off the neural network during training and stochastic Gradient Descent (SGD) is used to adjust the parameters based on the gradients calculated from the loss the learning rate is set to `0.001` what controls how large each update to the model parameters can be so momentum is set to `0.9`what helps the optimiser continue moving in useful directions and can make the training process more efficient a `StepLR` learning rate scheduler is also defined this changes the learning rate during training by reducing it according to the specified settings the `step_size` is set to 7 meaning the scheduler reduces the learning rate after 7 epochs, while `gamma=0.1` reduces the learning rate to 10% of its previous value

In [17]:
# Define the optimiser and learning rate scheduler
optimizer = optim.SGD(
    model.classifier.parameters(),
    lr=0.001,
    momentum=0.9
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=7,
    gamma=0.1
)

### Defining the Loss Function

The loss function is used to measure how accurately the model is making its predictions `CrossEntropyLoss()` is used because the CIFAR-10 dataset contains multiple different classes that the model needs to classify the loss compares the model's out put with the correct labels from the dataset a lower loss indicates that the model's predictions are closer to the correct answers then during training the loss is used with the gradients to help the optimiser update the trainable parameters of the classifier

In [18]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

### Setting Training and Early-Stopping Values

The number off training epochs is set to 5, meaning the model can make up to five passes through the training dataset the `best_val_loss` variable is initially set to infinity so that the first validation loss calculated during training will be considered an improvement the `patience` value is set to 3 this is used for early stopping what helps prevent unnecessary training if the validation loss stops improving the `trigger_times` variable starts at zero and is used to count consecutive times where the validation loss does not improve these settings help control the length off the training process and provide a way to stop training early when the model is no longer improving on the validation data

In [19]:
# Set the number of training epochs
num_epochs = 5

# Set the initial best validation loss
best_val_loss = float('inf')

# Set the number of epochs to wait before early stopping
patience = 3

# Keep track of how many times the validation loss has not improved
trigger_times = 0

### Training the Model

The training loop allows the model to learn from the training dataset over several epochs at the beginning off each epoch `model.train()` places the model into training mode the training data is then provided in batches using the training DataLoader the in put images and there labels are moved to the selected computing device andefore calculating new gradients, `optimizer.zero_grad()` clears any gradients left from the previous batch the model then processes the in put images and produces predictions the loss function compares these predictions with the correct labels `loss.backward()` calculates the gradients needed to determine how the trainable parameters should be adjusted the optimiser then uses these gradients with `optimizer.step()` to update the model parameters the loss from each batch is added to `running_loss` so that the training performance can be monitored

In [20]:
# Start the training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

###  Validating the Model 

After each training epoch the model is evaluated using the validation dataset `model.eval()` changes the model to evaluation mode because the model is no longer being trained at this point the `torch.no_grad()` context is used because gradients are not required when validating the model this reduces unnecessary calculations the validation images and labels are loaded in batches using `val_loader`, and the model produces predictions for each batch the loss function then compares the predictions with the correct validation labels and each batch's loss is added to `val_loss` plus the total is divided by the number off validation batches to calculate the average validation loss this value can then be used to monitor whether the model is improving during training

In [21]:
# Evaluate the model on the validation dataset
model.eval()
val_loss = 0.0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        val_loss += loss.item()

val_loss /= len(val_loader)

###  Displaying the Training and Validation Loss

This code displays the training and validation loss after each epoch the epoch number shows what stage off the training process the model has reached the training loss is calculated by dividing the total running loss by the number of training batches giving an average loss for that epoch the validation loss is also displayed so that the performance off the model on the validation dataset can be monitored then comparing the training and validation loss helps show whether the model is improving as training continues these values also help identify when the validation performance stops improving what is important for the early stopping process used in this training.


In [22]:
# Display the training and validation loss
print(
    f"Epoch {epoch+1}/{num_epochs}, "
    f"Training Loss: {running_loss/len(train_loader)}, "
    f"Validation Loss: {val_loss}"
)

Epoch 5/5, Training Loss: 0.4193810843064657, Validation Loss: 0.3479833775644849


### Updating the Learning Rate

The learning rate scheduler is updated at the end off each training epoch using `scheduler.step()` this allows the learning rate to change according to the settings defined earlier in the training process the `StepLR` scheduler reduces the learning rate based on the `step_size` and `gamma` values that were previously defined then adjusting the learning rate during training can help the model make smaller updates as training progresses

In [23]:
# Update the learning rate
scheduler.step()

## STEP 3) Data loading and checkpoints
In this step I tested the trained VGG16 model using the test dataset to measure how accurately it can classify unseen CIFAR-10 images the model was put into evaluation mode and torch.no_grad() was used so that gradients were not calculated during testing the predictions were compared with the correct labels to calculate the overall test accuracy I then saved the model's learned parameters as a checkpoint called vgg16_cifar10.pth finally I loaded the saved checkpoint back into the model and placed it into evaluation mode so that the trained model could be used again with out needing to retrain it

###  Loading the Best Model

The saved model checkpoint is loaded using `model.load_state_dict()` so during Step 2 the model was saved as `best_model.pth` when the validation loss improved loading this file restores the saved model parameters this means the model used for the next stage is the best version saved during training rather than simply the final version off the model the loaded model can then be used for making predictions on an image

In [24]:
torch.save(model.state_dict(), 'best_model.pth')

In [25]:
# Load the best model
model.load_state_dict(torch.load('best_model.pth'))

<All keys matched successfully>

### Loading the Test Image
A URL is provided for an image from the CIFAR-10 dataset the `requests.get()` function is used to retrieve the image from the specified URL the downloaded content is then passed to `BytesIO` what allows the image data to be treated as a file in memory `Image.open()` from the PIL library is then used to open the image and store it in the `img` variable this prepares the image so that it can be processed using the transformations and passed to the trained model for inference.


In [26]:
# Set the URL of the test image
url = "https://www.cs.toronto.edu/~kriz/cifar-10-sample/ship7.png"

# Download the image
response = requests.get(url)

# Open the downloaded image
img = Image.open(BytesIO(response.content))

###  Checking the Image Format

The image format is checked before it is processed by the model the `img.mode` property identifies the current image model the code checks if this is equal to `RGB` what is the colour format required for the image processing in this step if the image is not already in RGB format, `img.convert('RGB')` converts it to RGB this helps ensure that the image has a consistent three-channel colour format before the inference transformations are applied

In [27]:
# Convert the image to RGB if necessary
if img.mode != 'RGB':
    img = img.convert('RGB')

### Inference transformations
The inference transformations prepare the image so that it is in the correct format for the trained VGG16 model see first `Resize(256)` changes the image so that its smaller side is 256 pixels `CenterCrop(224)` then takes the central 224 × 224 section off the image what matches the in put size expected by VGG16. `ToTensor()` converts the image into a PyTorch tensor so that it can be processed by the model then finally `Normalize()` normalises the image using the same mean and standard deviation values used for the pre trained VGG16 model these transformations help ensure that the image is prepared consistently before it is passed to the model for inference

In [28]:
# Define the transformations for inference
transform_inference = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

### Convert the image into a tensor

The inference transformations are applied to the image using `transform_inference(img)` this carries out the resizing, centre cropping conversion to a tensor and normalisation that were defined in the previous step the result is stored in `img_t`, what is the transformed version off the image that can be used as in put for the PyTorch model

In [29]:
# Apply the inference transformations to the image
img_t = transform_inference(img)

### Add the batch dimension

`unsqueeze(0)` adds an extra dimension to the image tensor so that it represents a batch containing one image this is needed because the model expects its in put to include a batch dimension even when only one image is being classified. `.to(device)` then moves the tensor to the same device as the model, such as the GPU if CUDA is available or the CPU otherwise this makes the image ready to be passed into the model for inference.


In [30]:
# Add a batch dimension and move the image to the device
img_t = img_t.unsqueeze(0).to(device)

### Run inference

The model is changed to evaluation mode using `model.eval()` what makes sure it behaves correctly when classifying the image rather than training `torch.no_grad()` is used because the model does not need to calculate gradients during inference this reduces the amount off memory and processing required the transformed image is then passed into the model using `model(img_t)`, producing the model's output containing its predictions for the different CIFAR-10 classes


In [31]:
# Set the model to evaluation mode
model.eval()

# Run inference without calculating gradients
with torch.no_grad():
    output = model(img_t)

### Get the predicted class

`torch.max(output, 1)` finds the highest prediction score from the model's out put the position off this highest score represents the class that the model believes the image belongs to the predicted class is then stored in `predicted` the `.item()` method converts the single prediction from a PyTorch tensor into a normal Python number what can then be used to identify the corresponding CIFAR-10 class


In [32]:
# Get the class with the highest prediction score
_, predicted = torch.max(output, 1)

# Convert the predicted class to a Python number
predicted_class = predicted.item()

### CIFAR-10 class labels

The `class_labels` list contains the ten categories used by the CIFAR-10 dataset each class has a position in the list starting from 0 the predicted class number produced by the model can therefore be used as an index to find the name off the predicted category like if the predicted class is `8`, the corresponding label is `Ship` this makes the model's numerical prediction easier to understand.


In [33]:
# Define the CIFAR-10 class labels
class_labels = [
    'Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog',
    'Frog', 'Horse', 'Ship', 'Truck'
]

### Display the predicted class

The predicted class number is used to select the matching class name from the `class_labels` list the `print()` statement then displays the result in a clear and readable format this allows the final prediction from the trained VGG16 model to be understood as an actual CIFAR-10 category such as `Ship` rather than just being shown as a numerical value.

In [34]:
# Display the predicted class label
print(f'Predicted class: {class_labels[predicted_class]}')

Predicted class: Ship
